# Week 6 — Notebook 2: Logistic Regression

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2.5 hours  
**Theme:** Spam Email Detection (spam = 1, not-spam = 0)

---

## Why This Matters

Notebook 1 showed *what* a classifier does — it outputs probabilities and draws decision boundaries.  
This notebook answers *how*: where do those probabilities come from, and what equation produces them?

Logistic Regression is the **gateway drug** of classification:
- It is the foundation of understanding **neural network output layers**
- Its coefficients are directly interpretable in business terms (**odds ratios**)
- It is fast, scalable, and still used in production at Google, Facebook, and banks worldwide
- Understanding it deeply makes every subsequent classifier (trees, SVMs, neural nets) easier to grasp

---

## The Analogy: A Dimmer Switch, Not a Light Switch

Imagine you're classifying emails as spam or not-spam.  
A **light switch** (linear regression) gives you a binary on/off — but with emails, some are obviously spam (fully lit), some are clearly ham (fully off), and many are in between (dim).

The **sigmoid function** is a dimmer switch:  
- No matter how much you turn the dial (positive or negative infinity), the light never goes brighter than fully on (1) or fully off (0)
- In the middle, it transitions smoothly from dark to bright
- The transition happens at the switching point (z = 0), where P = 0.5

This smooth S-shaped curve is the heart of Logistic Regression.

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain **why linear regression fails** for classification tasks
2. Derive the **sigmoid function** and explain its mathematical properties
3. Connect **log-odds (logit)** to probability using the sigmoid
4. Interpret **coefficients as odds ratios** using e^θ
5. Verify that the logistic regression **decision boundary is linear**
6. Assess **probability calibration** using a reliability diagram

## Section 1 — Why Linear Regression Fails for Classification

### The Problem in Plain English

Suppose we force linear regression to predict a binary label (0 or 1):

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots$$

Two fatal problems arise:

1. **Unbounded outputs:** The model can predict 1.7 or -0.3. These are not valid probabilities.
2. **Sensitivity to outliers:** A single extreme email (10,000 words, 0 links, clearly ham) can tilt the decision boundary far from where it should be.

We need a function that:
- Takes any real number z ∈ (-∞, +∞) as input
- Always outputs a value in [0, 1]
- Is smooth and differentiable (so we can train with gradient descent)

Enter the **sigmoid function**.

In [ ]:
# ── Cell 1: Imports and global settings ──────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.linear_model   import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler
from sklearn.calibration    import CalibrationDisplay
from sklearn.metrics        import accuracy_score

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plot aesthetics
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print("Libraries loaded.")

In [ ]:
# ── Cell 2: Rebuild the spam dataset (same as Notebook 1) ─────────────────────
N, spam_frac = 2000, 0.20
n_spam = int(N * spam_frac)
n_ham  = N - n_spam

# Not-spam (ham) email features
ham = {
    'email_length'  : np.random.normal(150, 60,  n_ham).clip(10, 600),
    'num_links'     : np.random.poisson(1.5,      n_ham),
    'num_caps_words': np.random.poisson(0.5,      n_ham),
    'has_attachment': np.random.binomial(1, 0.10, n_ham),
    'label'         : np.zeros(n_ham)
}

# Spam email features
spam = {
    'email_length'  : np.random.normal(80,  40,  n_spam).clip(5, 300),
    'num_links'     : np.random.poisson(6.0,      n_spam),
    'num_caps_words': np.random.poisson(4.0,      n_spam),
    'has_attachment': np.random.binomial(1, 0.45, n_spam),
    'label'         : np.ones(n_spam)
}

df = pd.concat([pd.DataFrame(ham), pd.DataFrame(spam)], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

FEATURES = ['email_length', 'num_links', 'num_caps_words', 'has_attachment']
X = df[FEATURES].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Dataset: {df.shape[0]} emails | {n_spam} spam ({spam_frac*100:.0f}%) | {n_ham} ham ({(1-spam_frac)*100:.0f}%)")

## Section 2 — The Sigmoid Function: Derivation and Properties

### The Formula

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

where **z** is a weighted sum of features: $z = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots$

### Key Properties

| Property | Value | Meaning |
|----------|-------|---------|
| σ(0) | 0.5 | When features give no information either way, probability is 50% |
| σ(+∞) | 1.0 | Overwhelming evidence for spam → near certainty |
| σ(-∞) | 0.0 | Overwhelming evidence against spam → near certainty of ham |
| σ'(z) | σ(z)·(1-σ(z)) | Derivative is elegant — product of the output and its complement |
| σ'(0) | 0.25 | Maximum gradient at z=0 — the model learns fastest here |

The derivative property $\sigma'(z) = \sigma(z)(1-\sigma(z))$ is why backpropagation in neural networks is computationally efficient.

In [ ]:
# ── Cell 3: Plot sigmoid function and its derivative ──────────────────────────
z    = np.linspace(-8, 8, 400)
sig  = 1 / (1 + np.exp(-z))            # sigmoid
dsig = sig * (1 - sig)                  # derivative: σ(z)·(1-σ(z))

fig, ax = plt.subplots(figsize=(10, 5))

# Sigmoid curve
ax.plot(z, sig,  color='royalblue', lw=2.8, label=r'$\sigma(z) = \frac{1}{1+e^{-z}}$')
ax.fill_between(z, sig, alpha=0.10, color='royalblue')

# Derivative curve
ax.plot(z, dsig, color='tomato', lw=2.2, linestyle='--',
        label=r"$\sigma'(z) = \sigma(z)\cdot(1-\sigma(z))$")

# Key reference lines
ax.axhline(0.5, color='grey',  lw=1.2, linestyle=':')
ax.axhline(1.0, color='grey',  lw=1.0, linestyle=':')
ax.axhline(0.0, color='grey',  lw=1.0, linestyle=':')
ax.axvline(0.0, color='black', lw=1.2, linestyle='--', label='z = 0  →  σ = 0.5')

# Annotate key points
ax.scatter([0], [0.5],  s=120, color='royalblue', zorder=5)
ax.scatter([0], [0.25], s=120, color='tomato',    zorder=5)
ax.annotate('σ(0) = 0.5', xy=(0, 0.5),  xytext=(1.5, 0.52), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='royalblue'), color='royalblue')
ax.annotate("σ'(0) = 0.25 (max gradient)", xy=(0, 0.25), xytext=(1.5, 0.20),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='tomato'), color='tomato')

ax.set_xlabel('z  (log-odds / linear combination of features)', fontsize=12)
ax.set_ylabel('Output', fontsize=12)
ax.set_title('The Sigmoid Function — The Heart of Logistic Regression', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='center right')
ax.set_xlim(-8, 8)
ax.set_ylim(-0.05, 1.08)
plt.tight_layout()
plt.show()

print("The sigmoid maps any real number to (0, 1) — always a valid probability.")
print("The derivative is highest at z=0, meaning the model learns most where it is most uncertain.")

## Section 3 — Odds, Log-Odds, and Probability: The Three Representations

### The Probability → Odds → Log-Odds Chain

$$P \xrightarrow{\div (1-P)} \text{Odds} = \frac{P}{1-P} \xrightarrow{\ln} \text{Log-Odds} = \ln\frac{P}{1-P}$$

And the reverse (sigmoid):

$$\text{Log-Odds} = z \xrightarrow{\sigma} P = \frac{1}{1+e^{-z}}$$

| Representation | Range | Interpretation |
|----------------|-------|---------------|
| Probability P | [0, 1] | Direct: P(spam) = 0.8 → 80% chance spam |
| Odds | [0, +∞) | P/(1-P) = 4 → spam is 4× more likely than not-spam |
| Log-Odds (logit) | (-∞, +∞) | ln(4) ≈ 1.39 → convenient for linear modelling |

In [ ]:
# ── Cell 4: Odds ↔ Log-Odds ↔ Probability conversion table ───────────────────
probabilities = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

rows = []
for p in probabilities:
    odds     = p / (1 - p)          # P / (1 - P)
    log_odds = np.log(odds)          # ln(Odds)  =  logit(P)
    p_check  = 1 / (1 + np.exp(-log_odds))  # σ(log_odds) should equal p
    rows.append({
        'Probability P'     : f"{p:.2f}",
        'Odds P/(1-P)'      : f"{odds:.4f}",
        'Log-Odds ln(Odds)' : f"{log_odds:+.4f}",
        'σ(Log-Odds) check' : f"{p_check:.4f}"
    })

df_conversion = pd.DataFrame(rows)

print("=" * 65)
print("Odds ↔ Log-Odds ↔ Probability Conversion Table")
print("=" * 65)
print(df_conversion.to_string(index=False))
print()
print("Key observations:")
print("  • P=0.50 → Log-Odds=0.0  (sigmoid(0) = 0.5 — confirmed)")
print("  • P=0.75 → Odds=3.0      (spam 3× more likely than ham)")
print("  • P=0.10 → Log-Odds<0    (negative log-odds = evidence against spam)")
print("  • σ column equals P column — the math checks out.")

## Section 4 — Why Linear Regression Fails: A Concrete Demonstration

Here we fit a **LinearRegression** model to the binary spam labels.  
We will see that it produces predicted values outside [0, 1] — values like 1.3 or -0.2.  
These are meaningless as probabilities.

Then we compare against Logistic Regression, which is bounded by design.

In [ ]:
# ── Cell 5: Linear regression on binary labels — out-of-range predictions ─────
# Fit a linear regression to the binary target
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
lin_preds = lin_reg.predict(X_test)   # unconstrained predictions

# Logistic regression for comparison
log_reg = LogisticRegression(random_state=SEED, max_iter=1000)
log_reg.fit(X_train, y_train)
log_probs = log_reg.predict_proba(X_test)[:, 1]  # bounded [0,1]

# ── Report out-of-range predictions ──────────────────────────────────────────
n_above1 = (lin_preds > 1.0).sum()
n_below0 = (lin_preds < 0.0).sum()
print(f"Linear Regression predictions:")
print(f"  Min = {lin_preds.min():.4f}, Max = {lin_preds.max():.4f}")
print(f"  Values > 1.0 : {n_above1}  ({n_above1/len(lin_preds)*100:.1f}% of test set)")
print(f"  Values < 0.0 : {n_below0}  ({n_below0/len(lin_preds)*100:.1f}% of test set)")
print()
print(f"Logistic Regression probabilities:")
print(f"  Min = {log_probs.min():.4f}, Max = {log_probs.max():.4f}")
print(f"  Values > 1.0 : 0   (guaranteed by sigmoid)")
print(f"  Values < 0.0 : 0   (guaranteed by sigmoid)")

In [ ]:
# ── Cell 6: Side-by-side histogram: Linear vs. Logistic predictions ────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ── Left: Linear Regression ───────────────────────────────────────────────────
axes[0].hist(lin_preds[y_test == 0], bins=35, alpha=0.65, color='steelblue',
             label='Ham',  edgecolor='white')
axes[0].hist(lin_preds[y_test == 1], bins=35, alpha=0.65, color='tomato',
             label='Spam', edgecolor='white')
axes[0].axvline(0.0, color='red',   lw=2, linestyle='--', label='Probability = 0')
axes[0].axvline(1.0, color='green', lw=2, linestyle='--', label='Probability = 1')
axes[0].fill_between([-0.6, 0.0], 0, 80, alpha=0.10, color='red',
                     label='Invalid (< 0)')
axes[0].fill_between([1.0, 1.6], 0, 80, alpha=0.10, color='green',
                     label='Invalid (> 1)')
axes[0].set_title('Linear Regression — INVALID probabilities', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Value')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)

# ── Right: Logistic Regression ────────────────────────────────────────────────
axes[1].hist(log_probs[y_test == 0], bins=35, alpha=0.65, color='steelblue',
             label='Ham',  edgecolor='white')
axes[1].hist(log_probs[y_test == 1], bins=35, alpha=0.65, color='tomato',
             label='Spam', edgecolor='white')
axes[1].axvline(0.0, color='red',   lw=2, linestyle='--')
axes[1].axvline(1.0, color='green', lw=2, linestyle='--')
axes[1].set_title('Logistic Regression — VALID probabilities ∈ [0, 1]', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Probability P(spam)')
axes[1].set_ylabel('Count')
axes[1].legend(fontsize=8)

plt.suptitle('Why Linear Regression Fails for Binary Classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 5 — The Logistic Regression Model: Formula

### The Complete Model

**Step 1 — Linear combination (log-odds):**
$$z = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3 + \theta_4 x_4$$

where for our spam problem:
- $x_1$ = email_length
- $x_2$ = num_links
- $x_3$ = num_caps_words
- $x_4$ = has_attachment

**Step 2 — Sigmoid squashing:**
$$P(\text{spam} | \mathbf{x}) = \sigma(z) = \frac{1}{1 + e^{-z}}$$

**Step 3 — Decision:**
$$\hat{y} = \begin{cases} 1 & \text{if } P(\text{spam}|\mathbf{x}) \geq 0.5 \\ 0 & \text{otherwise} \end{cases}$$

### The Log-Odds (Logit) Interpretation

Taking the inverse sigmoid:
$$\ln\frac{P}{1-P} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots$$

The **left side is the log-odds** of spam. Logistic regression models log-odds as a **linear function** of features.  
Each coefficient $\theta_j$ represents the change in log-odds per unit increase in $x_j$.

In [ ]:
# ── Cell 7: Manual sigmoid prediction — recreate what sklearn does internally ──
# Extract fitted coefficients from the trained model
theta_0   = log_reg.intercept_[0]       # bias term
theta_1to4 = log_reg.coef_[0]           # feature coefficients

print("Fitted Logistic Regression Coefficients (on standardised features):")
print(f"  θ₀ (intercept)    = {theta_0:+.4f}")
for feat, coef in zip(FEATURES, theta_1to4):
    print(f"  θ ({feat:<20}) = {coef:+.4f}")
print()

# Manual prediction for the first 5 test emails
print("Manual Prediction for First 5 Test Emails:")
print("-" * 60)
for i in range(5):
    x_i   = X_test[i]                          # standardised feature vector
    z_i   = theta_0 + np.dot(theta_1to4, x_i)  # linear combination
    sigma_i = 1 / (1 + np.exp(-z_i))           # sigmoid activation
    sklearn_prob = log_probs[i]                  # sklearn's answer
    match = "OK" if abs(sigma_i - sklearn_prob) < 1e-6 else "MISMATCH"
    print(f"  Email {i+1}: z={z_i:+.4f}  →  σ(z)={sigma_i:.6f}  "
          f"[sklearn={sklearn_prob:.6f}]  [{match}]")

print()
print("Our manual sigmoid matches sklearn exactly — we've recreated the model from scratch!")

## Section 6 — Coefficient Interpretation: Odds Ratios

### The Formula

Since log-odds = θᵀx, each coefficient has a clean interpretation:

> A one-unit increase in feature $x_j$ **multiplies the odds of spam by $e^{\theta_j}$**.

This is called the **odds ratio** for feature j.

| Odds Ratio e^θⱼ | Interpretation |
|-----------------|---------------|
| e^θⱼ > 1 | Feature increases odds of spam |
| e^θⱼ = 1 | Feature has no effect on spam probability |
| e^θⱼ < 1 | Feature decreases odds of spam (protective factor) |

**Example:** If θ(num_links) = 0.8, then e^0.8 ≈ 2.23.  
Each additional link in an email **multiplies the odds of it being spam by 2.23×**.

Note: coefficients are on standardised features, so interpret relative magnitudes, not absolute units.

In [ ]:
# ── Cell 8: Compute and display odds ratios ────────────────────────────────────
coefs      = log_reg.coef_[0]
odds_ratios = np.exp(coefs)        # e^θ for each feature

# Build a readable summary DataFrame
df_coefs = pd.DataFrame({
    'Feature'     : FEATURES,
    'Coefficient θ': coefs.round(4),
    'Odds Ratio e^θ': odds_ratios.round(4),
    'Effect'      : ['Increases spam odds' if o > 1 else 'Decreases spam odds'
                     for o in odds_ratios]
}).sort_values('Odds Ratio e^θ', ascending=False).reset_index(drop=True)

print("=" * 65)
print("Logistic Regression: Coefficient Interpretation via Odds Ratios")
print("=" * 65)
print(df_coefs.to_string(index=False))
print()
print("Interpretation example:")
top = df_coefs.iloc[0]
print(f"  '{top['Feature']}' has odds ratio {top['Odds Ratio e^θ']:.3f}")
print(f"  → One SD increase in {top['Feature']} multiplies spam odds by {top['Odds Ratio e^θ']:.2f}×")

In [ ]:
# ── Cell 9: Horizontal bar chart of odds ratios ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))

# Sort by odds ratio for a clean chart
sorted_idx  = np.argsort(odds_ratios)
sorted_feats = np.array(FEATURES)[sorted_idx]
sorted_OR    = odds_ratios[sorted_idx]

colors = ['tomato' if o > 1 else 'steelblue' for o in sorted_OR]
bars = ax.barh(sorted_feats, sorted_OR, color=colors, edgecolor='white', height=0.55)

# Reference line at OR = 1.0 (no effect)
ax.axvline(1.0, color='black', lw=2.0, linestyle='--', label='OR = 1.0 (no effect)')

# Value annotations
for bar, val in zip(bars, sorted_OR):
    offset = 0.03 if val >= 1 else -0.15
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=11)

# Legend patches
import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='tomato',    label='Increases spam odds (OR > 1)'),
    mpatches.Patch(color='steelblue', label='Decreases spam odds (OR < 1)'),
    plt.Line2D([0],[0], color='black', lw=2, linestyle='--', label='No effect (OR = 1)')
], fontsize=10, framealpha=0.9)

ax.set_xlabel('Odds Ratio e^θ', fontsize=12)
ax.set_title('Odds Ratios — Which Features Drive Spam?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Features with OR > 1 push an email toward spam.")
print("Features with OR < 1 push an email toward ham (not-spam).")

## Section 7 — The Decision Boundary is Linear

### Mathematical Proof

The decision boundary is where P(spam|x) = 0.5:

$$\sigma(\theta^T x) = 0.5$$
$$\Rightarrow \theta^T x = \sigma^{-1}(0.5) = 0$$
$$\Rightarrow \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots = 0$$

This is a **linear equation** — the decision boundary is a **hyperplane** (a line in 2D, a plane in 3D).

**Consequence:** Logistic regression is a **linear classifier**.  
It cannot solve the XOR problem or any data that isn't linearly separable, without feature engineering.

We will now visualise this boundary in 2D (email_length × num_links).

In [ ]:
# ── Cell 10: 2D decision boundary — draw the line where θᵀx = 0 ───────────────
# Refit on 2 features only for 2D visualisation
FEAT2   = ['email_length', 'num_links']
X2      = df[FEAT2].values
y2      = df['label'].values
scaler2 = StandardScaler()
X2_sc   = scaler2.fit_transform(X2)

clf2 = LogisticRegression(random_state=SEED, max_iter=1000)
clf2.fit(X2_sc, y2)

# Extract coefficients for the 2-feature model
th0, th1, th2 = clf2.intercept_[0], clf2.coef_[0, 0], clf2.coef_[0, 1]

# ── Build mesh for P(spam) colouring ──────────────────────────────────────────
x_min, x_max = X2[:, 0].min() - 20, X2[:, 0].max() + 20
y_min, y_max = X2[:, 1].min() - 1,  X2[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                     np.linspace(y_min, y_max, 400))
grid_sc = scaler2.transform(np.c_[xx.ravel(), yy.ravel()])
ZZ = clf2.predict_proba(grid_sc)[:, 1].reshape(xx.shape)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
cf = ax.contourf(xx, yy, ZZ, levels=50, cmap='RdBu_r', alpha=0.70)
plt.colorbar(cf, ax=ax, label='P(spam | features)')

# Decision boundary: the black dashed contour at P=0.5 (i.e. θᵀx=0)
ax.contour(xx, yy, ZZ, levels=[0.5], colors='black', linewidths=2.8, linestyles='--')

# Scatter actual data
idx_h = np.where(y2 == 0)[0][:400]
idx_s = np.where(y2 == 1)[0][:400]
ax.scatter(X2[idx_h, 0], X2[idx_h, 1], s=20, color='steelblue', alpha=0.45, label='Ham')
ax.scatter(X2[idx_s, 0], X2[idx_s, 1], s=20, color='tomato',    alpha=0.60, label='Spam')

ax.set_xlabel('Email Length (words)', fontsize=12)
ax.set_ylabel('Number of Links', fontsize=12)
ax.set_title('Logistic Regression Decision Boundary (θᵀx = 0 → dashed line)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"2-feature model coefficients: θ₀={th0:.4f}, θ(length)={th1:.4f}, θ(links)={th2:.4f}")
print("Decision boundary equation: {:.4f} + {:.4f}·(scaled_length) + {:.4f}·(scaled_links) = 0".format(th0, th1, th2))

## Section 8 — Probability Calibration: Can We Trust the Probabilities?

### What is Calibration?

A **well-calibrated** model means: when it says P(spam) = 0.8, roughly 80% of those emails really are spam.  
A **poorly calibrated** model might say P = 0.8 but only 50% of those emails are spam.

**Why calibration matters:**
- In medical diagnosis: "70% chance of cancer" must actually mean 70%, not 30%
- In fraud detection: downstream risk scoring systems consume these probabilities directly

### The Reliability Diagram (Calibration Curve)

- **x-axis:** Mean predicted probability in each bin (what the model says)
- **y-axis:** Fraction of actual positives in that bin (ground truth)
- **Perfect calibration:** Points lie on the diagonal y = x
- Points **above** the diagonal → model is **underconfident** (actual rate > predicted)
- Points **below** the diagonal → model is **overconfident** (actual rate < predicted)

In [ ]:
# ── Cell 11: Calibration curve using sklearn CalibrationDisplay ────────────────
fig, ax = plt.subplots(figsize=(7, 6))

# Draw the calibration curve for logistic regression
CalibrationDisplay.from_estimator(
    log_reg,
    X_test,
    y_test,
    n_bins=10,                  # divide predictions into 10 probability bins
    ax=ax,
    name='Logistic Regression',
    color='royalblue',
    marker='o',
    markersize=8
)

# Perfect calibration reference line
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect Calibration (y=x)')

ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Fraction of Positives (Actual Spam Rate)', fontsize=12)
ax.set_title('Calibration Curve — Are the Probabilities Trustworthy?', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, framealpha=0.9)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()
plt.show()

print("Logistic regression is naturally well-calibrated for linearly separable data.")
print("Points close to the diagonal confirm: when the model says 0.8, ~80% of emails are spam.")

## Section 9 — Sigmoid Saturation: The Vanishing Gradient Problem

One subtlety worth understanding: the sigmoid **flattens** at extreme z values.  
When z is very large or very small, σ'(z) ≈ 0 — the gradient nearly vanishes.

For logistic regression this is fine — it just means the model is very confident and gradients are near zero.  
But in **deep neural networks** with many sigmoid layers, this causes the **vanishing gradient problem**, which is why modern networks use ReLU instead of sigmoid in hidden layers.

This gives you a preview of why sigmoid is used in *output* layers (classification) but not *hidden* layers.

In [ ]:
# ── Cell 12: Visualise sigmoid saturation zones ────────────────────────────────
z_wide = np.linspace(-10, 10, 500)
sig_wide  = 1 / (1 + np.exp(-z_wide))
dsig_wide = sig_wide * (1 - sig_wide)

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(z_wide, dsig_wide, color='darkorange', lw=2.5,
        label="σ'(z) = σ(z)·(1-σ(z)) — gradient magnitude")

# Shade saturation zones where gradient is near zero
ax.fill_between(z_wide, dsig_wide, where=(z_wide < -4), alpha=0.25,
                color='red', label='Saturation zone (gradient ≈ 0)')
ax.fill_between(z_wide, dsig_wide, where=(z_wide > 4),  alpha=0.25,
                color='red')

# Shade active learning zone
ax.fill_between(z_wide, dsig_wide, where=np.abs(z_wide) <= 4, alpha=0.15,
                color='green', label='Active learning zone (gradient > 0.01)')

ax.axhline(0.0,  color='grey', lw=1.0, linestyle=':')
ax.axhline(0.25, color='grey', lw=1.0, linestyle=':', label='Peak gradient = 0.25 at z=0')

ax.set_xlabel('z (log-odds)', fontsize=12)
ax.set_ylabel("Gradient σ'(z)", fontsize=12)
ax.set_title('Sigmoid Gradient — Active vs. Saturated Zones', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

print("The sigmoid gradient is nearly zero for |z| > 4.")
print("In deep networks, this causes vanishing gradients — why ReLU replaced sigmoid in hidden layers.")

## Section 10 — Full Model Summary: Putting It All Together

In [ ]:
# ── Cell 13: Full model summary with feature importances ──────────────────────
from sklearn.metrics import classification_report

y_pred_final = log_reg.predict(X_test)

print("=" * 65)
print("LOGISTIC REGRESSION — SPAM CLASSIFIER FULL REPORT")
print("=" * 65)
print()
print("── Model Equation ──────────────────────────────────────────")
print(f"  z = {log_reg.intercept_[0]:+.4f}")
for feat, coef in zip(FEATURES, log_reg.coef_[0]):
    print(f"    {coef:+.4f} × {feat}")
print(f"  P(spam) = σ(z) = 1 / (1 + exp(-z))")
print()
print("── Odds Ratios ─────────────────────────────────────────────")
for feat, coef in zip(FEATURES, log_reg.coef_[0]):
    print(f"  {feat:<22}: e^{coef:+.4f} = {np.exp(coef):.4f}")
print()
print("── Classification Report ───────────────────────────────────")
print(classification_report(y_test, y_pred_final,
                             target_names=['Not-Spam (0)', 'Spam (1)']))
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_final):.4f}")

In [ ]:
# ── Cell 14: Comprehensive 4-panel summary visualisation ──────────────────────
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc

y_prob_final = log_reg.predict_proba(X_test)[:, 1]
fpr, tpr, _  = roc_curve(y_test, y_prob_final)
roc_auc      = auc(fpr, tpr)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# ── Panel 1: Probability histogram ───────────────────────────────────────────
ax = axes[0, 0]
ax.hist(y_prob_final[y_test == 0], bins=30, alpha=0.65, color='steelblue', label='Ham')
ax.hist(y_prob_final[y_test == 1], bins=30, alpha=0.65, color='tomato',    label='Spam')
ax.axvline(0.5, color='black', lw=2, linestyle='--')
ax.set_title('P(spam) Distributions', fontweight='bold')
ax.set_xlabel('P(spam)')
ax.legend()

# ── Panel 2: Confusion matrix ────────────────────────────────────────────────
ax = axes[0, 1]
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_final,
    display_labels=['Ham', 'Spam'],
    cmap='Blues', ax=ax
)
ax.set_title('Confusion Matrix (threshold=0.5)', fontweight='bold')

# ── Panel 3: ROC curve ───────────────────────────────────────────────────────
ax = axes[1, 0]
ax.fill_between(fpr, tpr, alpha=0.18, color='royalblue', label=f'AUC = {roc_auc:.3f}')
ax.plot(fpr, tpr, color='royalblue', lw=2.5)
ax.plot([0,1],[0,1],'k--', lw=1.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(loc='lower right')

# ── Panel 4: Coefficient bar chart ───────────────────────────────────────────
ax = axes[1, 1]
coefs_4 = log_reg.coef_[0]
colors_4 = ['tomato' if c > 0 else 'steelblue' for c in coefs_4]
ax.barh(FEATURES, coefs_4, color=colors_4, edgecolor='white')
ax.axvline(0, color='black', lw=1.5)
ax.set_xlabel('Coefficient θ (standardised)')
ax.set_title('Feature Coefficients (θ > 0 → spam)', fontweight='bold')

fig.suptitle('Logistic Regression Summary — Spam Email Classifier', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 11 — Concept Summary

| Concept | Plain English |
|---------|---------------|
| **Why not Linear Regression?** | Outputs can be >1 or <0 — not valid probabilities |
| **Sigmoid σ(z)** | S-curve that maps ℝ → (0,1); σ(0)=0.5; smooth and differentiable |
| **Log-odds (logit)** | The linear combination z = θᵀx; logistic regression models this linearly |
| **Coefficient θⱼ** | A one-unit increase in xⱼ multiplies odds by e^θⱼ |
| **Odds ratio e^θⱼ** | >1 = feature increases spam risk; <1 = protective |
| **Decision boundary** | Where θᵀx = 0 → always a **linear** hyperplane |
| **Calibration** | Well-calibrated model: P=0.8 really means 80% of those are spam |
| **Vanishing gradient** | Sigmoid saturates at extremes — issue for deep networks, not single LR |

### The Logistic Regression Workflow

```
Raw features → StandardScaler → θᵀx (linear combination)
                                       ↓
                              σ(z) = 1/(1+e^-z)   ← sigmoid
                                       ↓
                           P(spam|x) ∈ (0,1)
                                       ↓
                         If P ≥ threshold → spam
                         If P < threshold → ham
```

---

## Self-Check Questions

Test your understanding. Try to answer each question before revealing the answer.

---

**Question 1:**  
The logistic regression coefficient for `num_links` is θ = 0.8.  
What does this mean for the odds of an email being spam?

<details>
<summary>Show Answer</summary>

**Odds ratio = e^0.8 ≈ 2.23**

This means: for each additional link in an email (one unit increase in `num_links`), **the odds of the email being spam are multiplied by 2.23**.

In plain English:
- An email with 3 links is 2.23× more likely to be spam than an identical email with 2 links
- An email with 5 links is 2.23² ≈ 4.97× more likely to be spam than one with 3 links

Note: this interpretation assumes the feature is on its original scale. If standardised (as in our model), the interpretation is "per one standard deviation increase in num_links."

**Key fact:** If θ > 0 → OR > 1 → feature increases spam odds.  
If θ < 0 → OR < 1 → feature decreases spam odds (protective).  
If θ = 0 → OR = 1 → feature has no effect.
</details>

---

**Question 2:**  
A linear regression model predicts a probability of 1.3 for an email.  
What specific problem does the sigmoid function fix, and how?

<details>
<summary>Show Answer</summary>

**Problem:** Linear regression is unbounded — it can output any real number including values greater than 1 or less than 0, which cannot be interpreted as probabilities.

**How sigmoid fixes it:**  
The sigmoid function $\sigma(z) = \frac{1}{1+e^{-z}}$ is mathematically **bounded between 0 and 1 for all inputs**:

- As z → +∞: e^{-z} → 0, so σ(z) → 1/(1+0) = 1
- As z → -∞: e^{-z} → +∞, so σ(z) → 1/(+∞) = 0
- For any finite z: 0 < σ(z) < 1

So even if the linear part θᵀx = 15 (which would give 1.3 raw), the sigmoid squashes it to σ(15) ≈ 0.9999997 — still a valid probability, just very close to 1.

**Additionally:** Sigmoid is monotonically increasing and differentiable everywhere, enabling gradient descent training.
</details>

---

**Question 3:**  
Logistic regression has a linear decision boundary.  
Can it correctly classify the XOR pattern? Why or why not?

<details>
<summary>Show Answer</summary>

**No, logistic regression cannot correctly classify the XOR pattern.**

**The XOR pattern:**  
XOR (exclusive OR) assigns label 1 to points (0,0) and (1,1), and label 0 to points (0,1) and (1,0) — or vice versa. The classes are arranged in a checkerboard pattern.

**Why logistic regression fails:**  
Logistic regression's decision boundary is always the equation θ₀ + θ₁x₁ + θ₂x₂ = 0 — a straight line in 2D. No straight line can separate the XOR pattern; you would need at least two separate lines.

**The XOR problem is NOT linearly separable** — you cannot draw a single straight line that puts class-1 on one side and class-0 on the other.

**Solutions:**  
1. **Feature engineering:** Add polynomial features (x₁², x₂², x₁x₂). The cross term x₁x₂ makes XOR linearly separable in the new feature space.
2. **Non-linear classifiers:** Decision Trees, Random Forests, SVMs with RBF kernel, or Neural Networks can solve XOR naturally.
3. **Neural network:** A single hidden layer with just 2 neurons and sigmoid activations can perfectly solve XOR — this is a classical example used to motivate multi-layer networks.
</details>